In [1]:
import os, sys, asyncio, pathlib, time
from dotenv import load_dotenv

repo_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (p / "training" / "__init__.py").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / "training" / ".env")
print("repo:", repo_root)
print("FIREWORKS_API_KEY:", "set" if os.environ.get("FIREWORKS_API_KEY") else "MISSING")
print("WANDB_ENTITY     :", os.environ.get("WANDB_ENTITY") or "(unset -> no metrics)")

repo: /Users/sinan/cookbook
FIREWORKS_API_KEY: set
WANDB_ENTITY     : fireworks-devrel


In [ ]:
# --- training config (PROPERLY-SIZED run; do NOT run until you've read the cost note) ---
BASE_MODEL      = "accounts/fireworks/models/glm-5p1"
TOKENIZER_MODEL = "zai-org/GLM-5.1"
TRAINING_SHAPE  = "accounts/fireworks/trainingShapes/glm-5p1-200k-lora"  # LoRA, 8xB300
LORA_RANK       = 16

# Hard slice. Base GLM-5.1 measured ~28-44% here (noisy) -> real RL headroom.
DIFFICULTY_MIN  = 7.0

# WHY bigger: the 256-row smoke exhausted at step 26 because the dynamic filter dropped
# 92-98% of prompt groups (per-problem, the 8 samples were usually unanimous -> zero GRPO
# advantage). To train for ~50+ real steps we need many more prompts to survive filtering.
TRAIN_ROWS      = 3000    # ~50-60 trainable steps after ~95% filter rejection
EVAL_ROWS       = 150     # bigger held-out set -> tighter confidence than n=50

COMPLETIONS_PER_PROMPT = 8     # raise to 16 to reduce constant-reward filtering (2x sampling cost)
PROMPT_GROUPS_PER_STEP = 8     # bigger batch -> steadier GRPO gradient
MAX_COMPLETION_TOKENS  = 8192
LEARNING_RATE          = 1.7e-5
KL_BETA                = 0.001

# Sampling is the bottleneck (~65% of wall-time). Each replica is a full 8xB300.
# More replicas -> proportionally faster rollouts AND proportionally more $$.
SAMPLER_REPLICAS = 2

# Eval: GLM-5.1 (MoE) is nondeterministic even at temp 0 (base swung 44%->28% on the SAME
# 50 rows). Average several samples/row to get a stable number.
EVAL_SAMPLES = 3

OUTPUT_MODEL_ID = "glm5p1-deepmath-rl-v2"   # bare name (gets created under your account)

# ---- COST / TIME (read before running) ----
# Trainer: 1x 8xB300.  Sampler deployment: SAMPLER_REPLICAS x 8xB300.
# So total GPUs ~ 8*(1+SAMPLER_REPLICAS) = 24 B300 at SAMPLER_REPLICAS=2.
# Rough wall-time: ~50 steps x ~5 min/step ~= 4-5 hours.
# Rough cost: tens of B300-hours -> on the order of hundreds to ~1-2k USD. Dial TRAIN_ROWS /
# SAMPLER_REPLICAS down for a cheaper run. THIS CELL DOES NOTHING UNTIL YOU RUN GO-LIVE.

In [3]:
from datasets import load_dataset

SYSTEM_PROMPT = ("You are a helpful math assistant. Solve the problem step by step, "
                 "showing your reasoning. Put your final answer inside \\boxed{}.")

def to_row(r):
    return {"messages": [{"role": "system", "content": SYSTEM_PROMPT},
                         {"role": "user", "content": r["question"]}],
            "ground_truth": str(r["final_answer"])}

# Stream + filter to the hard slice so we don't download all 103K rows.
_need = TRAIN_ROWS + EVAL_ROWS
_stream = load_dataset("zwhe99/DeepMath-103K", split="train", streaming=True)
_rows = []
for r in _stream:
    if r.get("difficulty") is not None and r["difficulty"] >= DIFFICULTY_MIN:
        _rows.append(to_row(r))
        if len(_rows) >= _need:
            break
eval_rows  = _rows[:EVAL_ROWS]     # held-out for before/after
train_rows = _rows[EVAL_ROWS:]     # training prompts
print(f"collected {len(_rows)} rows (difficulty >= {DIFFICULTY_MIN}) -> "
      f"train={len(train_rows)} eval={len(eval_rows)}")
print("example:", train_rows[0]["messages"][1]["content"][:90], "| ans:", train_rows[0]["ground_truth"])

/opt/homebrew/Caskroom/miniconda/base/envs/cookbook/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/zwhe99/DeepMath-103K/resolve/5cf055d1fe3d7a2eb19719ac020211469736ae44/data/train-00000-of-00010.parquet
Retrying in 1s [Retry 1/5].


collected 306 rows (difficulty >= 7.0) -> train=256 eval=50
example: Evaluate the double integral \( \iint \delta (ax^2+by-c) \, dx \, dy \). | ans: \infty


In [5]:
# Robust, hang-safe reward (bounded math_verify subprocess) from the deepmath example.
from training.examples.rl.deepmath.train_deepmath import deepmath_reward

for comp, gt in [("\\boxed{0}", "0"), ("\\boxed{\\frac{1}{2}}", "\\frac{1}{2}"), ("\\boxed{7}", "3")]:
    print(f"reward({comp!r}, gt={gt!r}) = {deepmath_reward(comp, {'ground_truth': gt})}")

reward('\\boxed{0}', gt='0') = 1.0
reward('\\boxed{\\frac{1}{2}}', gt='\\frac{1}{2}') = 1.0
reward('\\boxed{7}', gt='3') = 0.0


In [6]:
# Single-turn DeepMath rollout for the async RL recipe: sample once, score with
# deepmath_reward, pack into a RolloutRun (assistant tokens trained, prompt masked).
from training.examples.rl.vanilla_sampler import build_deployment_sampler
from training.utils.rl.rollout import (
    MessageTrajectoryAssembler, RolloutRun, RolloutSample, TITOTokenizer,
)

def make_deepmath_rollout_fn(setup):
    sampler = build_deployment_sampler(setup)
    sample_kwargs = dict(setup.sample_kwargs)
    tokenizer = setup.tokenizer

    async def rollout_fn(sample_prompt):
        messages = sample_prompt.get("messages")
        gt = sample_prompt.get("ground_truth")
        if not messages or gt is None:
            return None
        assembler = MessageTrajectoryAssembler(TITOTokenizer(tokenizer))
        prompt_tokens = assembler.prepare_next_input(messages)
        completions = await sampler.sample_with_prompt_tokens(prompt_tokens, n=1, **sample_kwargs)
        if not completions:
            return None
        c = completions[0]
        prompt_len = int(c.prompt_len)
        out_tokens = list(c.full_tokens[prompt_len:])
        out_lp = list(c.inference_logprobs or [])
        if getattr(c, "logprobs_echoed", False) and len(out_lp) == len(c.full_tokens):
            out_lp = out_lp[prompt_len:]
        if not out_tokens or len(out_lp) != len(out_tokens):
            return None
        text = getattr(c, "text", "") or tokenizer.decode(out_tokens)
        assembler.add_assistant_response(
            request_messages=messages,
            assistant_message={"role": "assistant", "content": text},
            prompt_token_ids=prompt_tokens, completion_token_ids=out_tokens,
            completion_logprobs=out_lp, finish_reason=getattr(c, "finish_reason", "stop"),
        )
        reward = deepmath_reward(text, {"ground_truth": gt})
        tokens, logprobs, loss_mask = assembler.trajectory.to_flat()
        return RolloutRun(segments=[RolloutSample(tokens=tokens, logprobs=logprobs,
                                                  loss_mask=loss_mask, reward=reward)])
    return rollout_fn

In [ ]:
import litellm
litellm.drop_params = True

async def eval_model(model_str, rows, n_samples=1, temperature=0.0,
                     max_tokens=MAX_COMPLETION_TOKENS, concurrency=8, timeout=600):
    """Eval a serverless/deployed model. n_samples>1 averages per-row reward to wash out
    GLM MoE nondeterminism; temperature>0 recommended when n_samples>1."""
    sem = asyncio.Semaphore(concurrency)
    async def one_call(msgs, gt):
        async with sem:
            try:
                resp = await litellm.acompletion(model=model_str, messages=msgs,
                    temperature=temperature, max_tokens=max_tokens, timeout=timeout)
                return deepmath_reward(resp.choices[0].message.content or "", {"ground_truth": gt})
            except Exception:
                return None
    async def one_row(r):
        outs = [await_ for await_ in await asyncio.gather(
            *[one_call(r["messages"], r["ground_truth"]) for _ in range(n_samples)])]
        outs = [o for o in outs if o is not None]
        return sum(outs) / len(outs) if outs else None
    per_row = [s for s in await asyncio.gather(*[one_row(r) for r in rows]) if s is not None]
    return (sum(per_row) / len(per_row) if per_row else 0.0), len(per_row)

In [8]:
# BEFORE: base GLM-5.1 on the held-out hard slice (serverless, cheap).
base_acc, n = await eval_model("fireworks_ai/" + BASE_MODEL, eval_rows)
print(f"BEFORE (base glm-5p1): {base_acc:.1%}  ({n}/{len(eval_rows)} graded)")
print("If this isn't ~10-50%, raise/lower DIFFICULTY_MIN in the config cell and re-run from there.")

14:46:17 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:46:17 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



14:56:21 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:21 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:25 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:25 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:41 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:41 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:44 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:56:44 [INFO] 
LiteLLM completion() model= accounts/fireworks/models/glm-5p1; provider = fireworks_ai
14:57:11 - LiteLLM:INFO: utils.py:3293 - 
LiteLLM completion() model


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

BEFORE (base glm-5p1): 43.5%  (46/50 graded)
If this isn't ~10-50%, raise/lower DIFFICULTY_MIN in the config cell and re-run from there.


In [ ]:
# ⚠️ GO LIVE: provisions an 8xB300 LoRA trainer + colocated deployment and runs async GRPO.
import nest_asyncio; nest_asyncio.apply()   # Jupyter has a running loop; main() calls asyncio.run

from training.recipes.async_rl_loop import Config, main
from training.utils import DeployConfig, TrainerConfig, WandBConfig

cfg = Config(
    log_path="./deepmath_glm5p1_logs",
    base_model=BASE_MODEL,
    learning_rate=LEARNING_RATE,
    kl_beta=KL_BETA,
    lora_rank=LORA_RANK,
    completions_per_prompt=COMPLETIONS_PER_PROMPT,
    max_completion_tokens=MAX_COMPLETION_TOKENS,
    temperature=1.0,
    epochs=1,
    max_rows=len(train_rows),
    prompt_groups_per_step=PROMPT_GROUPS_PER_STEP,
    max_head_offpolicy_versions=0,
    output_model_id=OUTPUT_MODEL_ID,
    trainer=TrainerConfig(training_shape_id=TRAINING_SHAPE),
    deployment=DeployConfig(tokenizer_model=TOKENIZER_MODEL, replica_count=SAMPLER_REPLICAS),
    wandb=WandBConfig(entity=os.environ.get("WANDB_ENTITY", ""),
                      project=os.environ.get("WANDB_PROJECT", "deepmath-glm5p1"),
                      run_name=f"deepmath-glm5p1-{int(time.time()) % 100000}"),
)
# Drop prompt groups whose samples all share one reward (zero GRPO advantage -> no-op step).
result = main(
    cfg,
    rollout_fn_factory=make_deepmath_rollout_fn,
    rows=train_rows,
    dynamic_filter_fn=lambda pg: len(set(pg.rewards)) > 1,
)
print(result)

In [ ]:
# AFTER: deploy the trained adapter, then eval BOTH models on the same held-out rows.
#
# HOW TO DEPLOY (console is simplest): Deployments -> Create Deployment ->
#   Base Model = your trained model (glm5p1-deepmath-rl-v2)  [deploy the MODEL directly;
#   do NOT use base glm-5p1 + "enable addons" -- that fails to init on the RFT shape]
#   Shape = Full Precision (B300 FP8)   [matches the FP8 serverless base -> fair comparison]
#   Then set Min Replicas = 1 (scale-to-zero returns "no healthy upstream" mid-eval).
# Put the resulting deployment id below (NOT the bare model name -- that hits serverless,
# which can't serve a LoRA addon).
TRAINED_DEPLOYMENT = None   # e.g. "accounts/pyroworks/deployments/<id>"

if TRAINED_DEPLOYMENT:
    trained_m = "fireworks_ai/" + TRAINED_DEPLOYMENT
    base_m    = "fireworks_ai/" + BASE_MODEL
    # Re-measure BOTH with sample-averaging so the delta isn't MoE noise.
    (b_acc, b_n), (t_acc, t_n) = await asyncio.gather(
        eval_model(base_m,    eval_rows, n_samples=EVAL_SAMPLES, temperature=0.6),
        eval_model(trained_m, eval_rows, n_samples=EVAL_SAMPLES, temperature=0.6),
    )
    print(f"BASE    : {b_acc:.1%}  ({b_n}/{len(eval_rows)} rows, {EVAL_SAMPLES} samples each)")
    print(f"TRAINED : {t_acc:.1%}  ({t_n}/{len(eval_rows)} rows, {EVAL_SAMPLES} samples each)")
    print(f"DELTA   : {t_acc - b_acc:+.1%}")
    print("Delete the deployment when done so it stops billing.")
else:
    print("Deploy the trained model (see comment), set TRAINED_DEPLOYMENT, then run this cell.")